<a href="https://colab.research.google.com/github/habibur2716-commits/AI-Lab/blob/main/cafe_assistant_for_Chai.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import json
import urllib.request
import urllib.error
import getpass

MODEL = "gemini-3.5-flash-lite"          # error aaye to "gemini-3.5-flash" likhein

try:
    from google.colab import userdata
    API_KEY = userdata.get("GEMINI_API_KEY")
except Exception:
    API_KEY = getpass.getpass("Apni Gemini API key paste karein aur Enter dabayein: ")

print("Ready")

Ready


In [ ]:
def load_notes(filename):
    notes = []
    with open(filename, encoding="utf-8") as f:
        for line in f:
            line = line.strip()          # line ke aage peeche ki space hatao
            if line:                     # khali line chhod do
                notes.append(line)
    return notes

try:
    NOTES = load_notes("knowledge.txt")
    print(len(NOTES), "notes loaded from file")
except FileNotFoundError:
    print("knowledge.txt nahi mili! Pehle file upload karein (Files panel mein).")

6 notes loaded from file


In [ ]:
# aam lafz jo match nahi karne chahiye
SKIP = {"a", "the", "is", "are", "of", "to", "in", "on", "at", "for", "do", "you", "what", "how",
        "and", "it", "has", "have", "with", "your", "me", "i", "can", "kya", "hai", "ke", "ki",
        "ka", "liye", "se", "mein", "aur", "ho", "hain", "tak", "ko"}

def words(text):
    text = text.lower()
    for ch in ".,?!:;-()":
        text = text.replace(ch, " ")
    return {w for w in text.split() if w not in SKIP}

def retrieve(question, k=2):
    q = words(question)
    scored = []
    for note in NOTES:
        score = len(q & words(note))      # kitne lafz common hain
        if score > 0:
            scored.append((score, note))
    scored.sort(reverse=True)
    return [note for score, note in scored[:k]]

print(retrieve("What is the price of karak chai?"))

['Hot drinks: Doodh Patti Chai Rs 150, Karak Chai Rs 180, Latte Rs 400, Cappuccino Rs 420.', 'Chai and Chapters is a cozy cafe in Lahore. It has a reading corner with free books.']


In [ ]:
# SYSTEM PROMPT = bot ke rules
SYSTEM_PROMPT = """You are a friendly cafe assistant for Chai and Chapters Cafe.
Rules:
1. If the CONTEXT contains the answer, answer using ONLY the context and start with "[From file]".
2. If the CONTEXT is empty or does not contain the answer, answer from your own general knowledge and start with "[From Gemini]".
3. Never invent details about this cafe (prices, timings, menu). If it is a question about this cafe and the context lacks it, say: "Sorry, I do not have this information about the cafe."
4. Keep the answer short (maximum 3 sentences).
5. The customer may write in any language (English, Urdu or Roman Urdu). Understand the question in that language, but ALWAYS reply in English only."""

# USER PROMPT = har sawal ke saath context + sawal
USER_PROMPT = """CONTEXT:
{context}

QUESTION:
{question}"""

In [ ]:
def ask_gemini(system_prompt, user_prompt):
    url = f"https://generativelanguage.googleapis.com/v1beta/models/{MODEL}:generateContent"
    body = {
        "system_instruction": {"parts": [{"text": system_prompt}]},
        "contents": [{"role": "user", "parts": [{"text": user_prompt}]}],
    }
    request = urllib.request.Request(
        url,
        data=json.dumps(body).encode("utf-8"),
        headers={"Content-Type": "application/json", "x-goog-api-key": API_KEY},
    )
    try:
        with urllib.request.urlopen(request) as response:
            data = json.loads(response.read())
        return data["candidates"][0]["content"]["parts"][0]["text"]
    except urllib.error.HTTPError as e:
        return f"Error {e.code}: {e.read().decode()[:300]}"

def answer(question):
    found = retrieve(question)                          # 1. file ke notes mein dhoondo
    if found:
        context = "\n".join(found)
    else:
        context = "(no matching notes found)"           # kuch nahi mila
    user_prompt = USER_PROMPT.format(context=context, question=question)   # 2. user prompt
    return ask_gemini(SYSTEM_PROMPT, user_prompt)       # 3. Gemini

In [ ]:
print("Chatbot ready! Type exit to stop.")
while True:
    q = input("You: ")
    if q.lower() == "exit":
        break
    print("Bot:", answer(q), "\n")

Chatbot ready! Type exit to stop.
You: exit
